# T16 — Building a Paleomagnetic Reference Frame from the Global Paleomagnetic Database

Plate models like **Zahirovic2022** ship with two absolute reference frames, paleomagnetic and mantle. This notebook illustrates how a paleomagnetic reference frame is constructed, and how sensitive it is to the underlying data. In this tutorial we build a *paleomagnetic* reference frame from scratch: we load ~9,000 published paleomagnetic poles from the Global Paleomagnetic Database (GPMDB), quality-filter them, assign them to the plates of the Zahirovic2022 model, rotate them into the African plate frame using the model's own relative rotations, average them into an apparent polar wander path (APWP) with uncertainties, and finally convert that path into absolute plate rotations — embedded back into the model's rotation file as a new anchor plate (`701702`), alongside the model's built-in paleomagnetic frame (`701701`).

A paleomagnetic frame is **not** a mantle frame: it constrains true *paleolatitude* and plate orientation relative to the spin axis (paleolongitude is fundamentally unconstrained because the time-averaged geomagnetic field is radially symmetric), and it *includes true polar wander* — so it does not describe plate motion relative to the mantle.

## Learning objectives

- Parse the GPMDB and apply Van der Voo (1990)-style quality filtering as a *free parameter*.
- Assign paleopoles to the plates of a GPlately plate model with static polygons.
- Understand and apply Fisher (1953) statistics: how individual virtual geomagnetic poles (VGPs) and their uncertainties are combined into mean poles with A95 confidence circles.
- Visualise VGP clusters, APWPs, quality sensitivity and APW rates with pyGMT.
- Convert an APWP into absolute rotations and embed them as a new anchor plate in a rotation file.

## Prerequisites and runtime

- `gplately` (with `plate_model_manager`), `pygmt`, `pygplates`, `pandas`, plus `pip install access-parser`.
- The GPMDB v4.6 snapshot in `data/gpmdb_46/GD2004C.MDB` (included).
- Runtime: ~3–5 minutes (first run downloads Zahirovic2022 via the plate model manager).


In [ ]:
# Cell 1 — imports, library versions, plot styling
import math
import numpy as np
import pandas as pd
import pygplates
import pygmt
import gplately
from plate_model_manager import PlateModelManager

for mod in (np, pd, pygplates, pygmt, gplately):
    print(f"{mod.__name__:20s} {getattr(mod, '__version__', 'n/a')}")

# Plot styling: large, legible fonts for titles, axis labels, annotations and
# colour bars (GMT defaults are far too small for teaching figures).
pygmt.config(FONT_TITLE="18p,Helvetica-Bold,black",
             FONT_LABEL="14p,Helvetica,black",
             FONT_ANNOT_PRIMARY="12p,Helvetica,black")


## 1. Load the GPMDB and score data quality

GPMDB v4.6 (Pisarevsky, 2005) is an MS Access database; `access_parser` reads it in pure Python. We join `PMAGRESULT` (poles) with `ROCKUNIT` (sampling localities). A95 comes from `ED95` where present, else from the dp/dm error oval.

**What the quality score Q means.** Following Van der Voo (1990), each pole earns one point per criterion met (0–5 here):

1. **Age control** — magnetisation age constrained to within 40 Myr;
2. **Statistics** — at least 24 samples, precision k ≥ 10, A95 ≤ 16°;
3. **Demagnetisation** — adequate laboratory cleaning (DEMAGCODE ≥ 2);
4. **Field tests** — a positive fold, contact or conglomerate test (proves the magnetisation predates deformation/reheating);
5. **Reversals** — both polarities recorded (averages out secular variation and some overprints).

So **Q ≥ 2** is a permissive cut (most poles pass, including poorly dated or statistically weak ones), **Q ≥ 3** is the moderate standard we default to, and **Q ≥ 4** keeps only well-dated, statistically robust, field-tested data — far fewer poles, especially in the Paleozoic. The threshold is a *free parameter* of this workflow, and we will quantify below how strongly the resulting reference frame depends on it.


In [ ]:
# Cell 2 — load GPMDB and compute Q
from access_parser import AccessParser

db = AccessParser("data/gpmdb_46/GD2004C.MDB")
pm = pd.DataFrame(db.parse_table("PMAGRESULT"))
ru = pd.DataFrame(db.parse_table("ROCKUNIT"))
df = pm.merge(ru[["ROCKUNITNO", "ROCKNAME", "LOWAGE", "HIGHAGE"]], on="ROCKUNITNO")

poles = pd.DataFrame({
    "site_lat": pd.to_numeric(df["SLAT"], errors="coerce"),
    "site_lon": pd.to_numeric(df["SLONG"], errors="coerce"),
    "pole_lat": pd.to_numeric(df["PLAT"], errors="coerce"),
    "pole_lon": pd.to_numeric(df["PLONG"], errors="coerce"),
    "n_sites": pd.to_numeric(df["B"], errors="coerce"),
    "n_samples": pd.to_numeric(df["N"], errors="coerce"),
    "k": pd.to_numeric(df["KD"], errors="coerce"),
    "demag": pd.to_numeric(df["DEMAGCODE"], errors="coerce"),
    "tests": df["TESTS"].astype(str),
})
ed95 = pd.to_numeric(df["ED95"], errors="coerce")
dp, dm = pd.to_numeric(df["DP"], errors="coerce"), pd.to_numeric(df["DM"], errors="coerce")
poles["a95"] = ed95.where(ed95.notna() & (ed95 > 0), np.sqrt(dp * dm))
lo = pd.to_numeric(df["LOMAGAGE"], errors="coerce").fillna(pd.to_numeric(df["LOWAGE"], errors="coerce"))
hi = pd.to_numeric(df["HIMAGAGE"], errors="coerce").fillna(pd.to_numeric(df["HIGHAGE"], errors="coerce"))
poles["lo_age"], poles["hi_age"], poles["mid_age"] = lo, hi, 0.5 * (lo + hi)
poles = poles.dropna(subset=["site_lat", "site_lon", "pole_lat", "pole_lon", "mid_age"])

q = ((poles["hi_age"] - poles["lo_age"]) <= 40).astype(int)
q += ((poles["n_samples"].fillna(0) >= 24) & (poles["k"].fillna(0) >= 10) & (poles["a95"].fillna(99) <= 16)).astype(int)
q += (poles["demag"].fillna(0) >= 2).astype(int)
q += poles["tests"].apply(lambda t: any(c in t for c in ("F+", "Fo", "C+", "Co", "G+", "R+"))).astype(int)
q += poles["tests"].str.contains("R\+").astype(int)
poles["q"] = q
for qq in (2, 3, 4):
    print(f"Q >= {qq}: {(poles.q >= qq).sum()} poles")


## 2. Assign poles to Zahirovic2022 plates and select stable plates

Each pole is assigned a plate ID by point-in-polygon of its *present-day sampling site* in the model's static polygons (fetched by GPlately's plate model manager). We keep only poles from large, "stable" plates — small terranes record local block rotations, not the motion of major plates.


In [ ]:
# Cell 3 — plate model via the plate model manager; assign plate IDs
pmm = PlateModelManager()
model = pmm.get_model("Zahirovic2022", data_dir="./gplately_data")
rotation_model = pygplates.RotationModel(model.get_rotation_model())
sp = model.get_static_polygons()
static_polygons = pygplates.FeatureCollection(sp[0] if isinstance(sp, list) else sp)

partitioner = pygplates.PlatePartitioner(static_polygons, rotation_model)
cache = {}
def plate_of(lat, lon):
    key = (round(lat, 2), round(lon, 2))
    if key not in cache:
        part = partitioner.partition_point(pygplates.PointOnSphere(*key))
        cache[key] = part.get_feature().get_reconstruction_plate_id() if part else None
    return cache[key]

poles["plate_id"] = [plate_of(la, lo) for la, lo in zip(poles.site_lat, poles.site_lon)]
poles = poles.dropna(subset=["plate_id"])

areas = {}
for f in static_polygons:
    pid = f.get_reconstruction_plate_id()
    for g in f.get_geometries():
        try: areas[pid] = areas.get(pid, 0.0) + g.get_area()
        except AttributeError: pass
stable = {pid for pid, a in areas.items() if a >= 0.05}   # >= ~2 million km^2
poles = poles[poles.plate_id.isin(stable)]
print(len(poles), "poles on", poles.plate_id.nunique(), "stable plates")


### Where do the poles come from? (GPlately context map)

We use GPlately's `PlateReconstruction` + `PlotTopologies` with the `PygmtPlotEngine` to draw present-day coastlines, and overlay the quality-filtered sampling sites coloured by plate — the cratons doing the work become obvious at a glance.


In [ ]:
# Cell 4 — present-day map of sampling sites on stable plates
recon = gplately.PlateReconstruction(
    rotation_model=model.get_rotation_model(),
    topology_features=model.get_topologies(),
    static_polygons=model.get_static_polygons(),
)
gplot = gplately.PlotTopologies(recon, coastlines=model.get_coastlines(), time=0)
engine = gplately.PygmtPlotEngine()

q_ok = poles[poles.q >= 3]
fig = pygmt.Figure()
fig.basemap(region="d", projection="R0/26c",
            frame=["af", f"+tGPMDB sampling sites on stable plates (Q>=3, n={len(q_ok)})"])
engine.plot_geo_data_frame(fig, gplot.get_coastlines(), pen="0.2p,gray30", fill="gray95")
colors = ["steelblue", "firebrick", "seagreen", "darkorange", "mediumpurple",
          "goldenrod", "deeppink", "cadetblue", "olivedrab", "chocolate"]
for i, (pid, grp) in enumerate(q_ok.groupby("plate_id")):
    fig.plot(x=grp.site_lon, y=grp.site_lat, style="c0.10c",
             fill=colors[i % len(colors)], pen="0.1p,gray40")
fig.show(width=1100)


## 3. How VGPs are averaged: Fisher statistics

Each paleomagnetic pole is a **virtual geomagnetic pole (VGP)**: the position the north geomagnetic pole would have had, computed from the direction of remanent magnetisation at one rock unit. To turn many VGPs into an APWP we need three steps per time window:

1. **Rotate into a common frame.** A VGP from, say, North America is only comparable with one from Africa after both are expressed in the same plate frame. We rotate every VGP into the **African plate frame** using the plate model's *relative* rotation (pole's plate → Africa) at the pole's mean magnetisation age. Relative rotations are independent of the model's absolute frame, so this step is frame-agnostic.

2. **Resolve polarity.** The geomagnetic field reverses, so a VGP and its antipode are equally valid. Within each window we iteratively flip any VGP that lies more than 90° from the provisional mean until the cluster is self-consistent.

3. **Fisher averaging.** Treating each VGP as a unit vector **v**ᵢ on the sphere, the mean pole is the direction of the resultant **R** = Σwᵢ**v**ᵢ. The tightness of the cluster gives the precision parameter κ ≈ (N−1)/(N−R) and the **95% confidence circle**

   A95 ≈ acos[ 1 − ((N−R)/R)·(20^(1/(N−1)) − 1) ],

   the spherical analogue of a standard error of the mean: the *true* mean pole lies within A95 of the estimate with 95% confidence. Note the philosophy of the classic running mean: **the uncertainty of the mean comes from the scatter *between* poles, not from the individual A95s** — between-pole scatter (different rock units, ages within the window, unremoved overprints) almost always dominates. The individual uncertainties can still *enter as weights* wᵢ:

   - `fisher` — all poles weighted equally (the classic choice; robust, transparent);
   - `site_weighted` — wᵢ = number of sites Bᵢ (capped at 50), approximating the site-level VGP averaging philosophy of Vaes et al. (2023): well-sampled studies count more;
   - `inv_a95` — wᵢ = 1/A95ᵢ², classical precision weighting: tightly determined poles count more.

   For weighted means the effective sample size N_eff = (Σw)²/Σw² replaces N in the A95 formula.

The three worked examples below (40, 100 and 300 Ma) show exactly this: the individual VGPs of one window, with their own A95 circles, and the resulting mean VGP with its (much smaller) A95.


In [ ]:
# Cell 5 — user variables; rotate VGPs into the African plate frame; Fisher machinery
MOTHER_PLATE = 701   # Africa
Q_MIN   = 3          # quality threshold (free parameter — compare 2/3/4 below)
WINDOW  = 20         # Myr averaging window
STEP    = 10         # Myr between window centres
SCHEME  = "fisher"   # 'fisher' | 'site_weighted' | 'inv_a95'
START_AGE, END_AGE = 410, 0

def xyz(lat, lon):
    la, lo = np.radians(lat), np.radians(lon)
    return np.column_stack([np.cos(la)*np.cos(lo), np.cos(la)*np.sin(lo), np.sin(la)])

def fisher_mean(v, w):
    w = w / w.sum()
    Rv = (v * w[:, None]).sum(axis=0); R = np.linalg.norm(Rv); m = Rv / R
    n_eff = 1.0 / (w**2).sum()
    a95, kappa = np.nan, np.inf
    if n_eff > 2 and R < 1:
        kappa = (n_eff - 1) / (n_eff * (1 - R))
        arg = 1 - ((n_eff - R*n_eff)/(R*n_eff)) * ((1/0.05)**(1/(n_eff-1)) - 1)
        a95 = math.degrees(math.acos(np.clip(arg, -1, 1)))
    return (math.degrees(math.asin(m[2])), math.degrees(math.atan2(m[1], m[0])), a95, kappa, n_eff)

def small_circle(lat, lon, radius_deg, n=73):
    c = xyz([lat], [lon])[0]
    ref = np.array([0, 0, 1.0]) if abs(c[2]) < 0.95 else np.array([1.0, 0, 0])
    u = np.cross(c, ref); u /= np.linalg.norm(u); v2 = np.cross(c, u)
    t = np.linspace(0, 2*np.pi, n); r = math.radians(radius_deg)
    p = math.cos(r)*c[:, None] + math.sin(r)*(np.cos(t)*u[:, None] + np.sin(t)*v2[:, None])
    return np.degrees(np.arcsin(p[2])), np.degrees(np.arctan2(p[1], p[0]))

def rotate_to_africa(s):
    la, lo = [], []
    for _, r in s.iterrows():
        rot = rotation_model.get_rotation(float(r.mid_age), int(r.plate_id), fixed_plate_id=MOTHER_PLATE)
        a, b = (rot * pygplates.PointOnSphere(r.pole_lat, r.pole_lon)).to_lat_lon()
        la.append(a); lo.append(b)
    s = s.copy(); s["plat"], s["plon"] = la, lo
    return s

def window_vgps(s, age, window=WINDOW):
    return s[(s.mid_age >= age - window/2) & (s.mid_age <= age + window/2)]

def weights_for(w_sel, scheme):
    if scheme == "site_weighted": return np.clip(w_sel.n_sites.fillna(1).values, 1, 50)
    if scheme == "inv_a95":       return 1.0 / w_sel.a95.fillna(20).clip(lower=1).values**2
    return np.ones(len(w_sel))

def mean_vgp(w_sel, scheme=SCHEME):
    v = xyz(w_sel.plat.values, w_sel.plon.values); w = weights_for(w_sel, scheme)
    for _ in range(10):                       # resolve antipodal polarity
        m = (v * (w/w.sum())[:, None]).sum(axis=0); m /= np.linalg.norm(m)
        flip = (v @ m) < 0
        if not flip.any(): break
        v[flip] *= -1
    mlat, mlon, a95, kappa, n_eff = fisher_mean(v, w)
    if mlat < 0:
        mlat, mlon, v = -mlat, (mlon + 180) % 360 - 180, -v
    return mlat, mlon, a95, kappa, v

vgps = {qm: rotate_to_africa(poles[(poles.q >= qm) & (poles.mid_age <= START_AGE)])
        for qm in (2, 3, 4)}
print({qm: len(v) for qm, v in vgps.items()})


### Worked examples: VGP clusters and their means at 40, 100 and 300 Ma

Each panel shows the individual VGPs (blue, with their own thin A95 circles), the Fisher mean (red star) with its bold A95 circle, and the geographic north pole (+). Notice how the mean's A95 is much smaller than the spread of individual poles — and how the cluster walks away from the geographic pole with age (apparent polar wander).


In [ ]:
# Cell 6 — VGP clusters at three ages (Q>=3), side by side
pygmt.config(FONT_TITLE="26p,Helvetica-Bold,black", FONT_ANNOT_PRIMARY="15p,Helvetica,black")
EXAMPLE_AGES = [40, 100, 300]
fig = pygmt.Figure()
for j, age in enumerate(EXAMPLE_AGES):
    if j > 0:
        fig.shift_origin(xshift="13.5c")
    w_sel = window_vgps(vgps[Q_MIN], age)
    mlat, mlon, a95, kappa, v = mean_vgp(w_sel)
    vlat = np.degrees(np.arcsin(v[:, 2])); vlon = np.degrees(np.arctan2(v[:, 1], v[:, 0]))
    fig.basemap(region="g", projection=f"G{mlon}/{max(mlat, 45)}/12c",
                frame=["g30", f"+t{age}\261{WINDOW//2} Ma: n={len(w_sel)}, A95={a95:.1f}\260"])
    fig.coast(land="gray95", shorelines="0.2p,gray60")
    for la, lo, a in zip(vlat, vlon, w_sel.a95.fillna(np.nan)):
        if np.isfinite(a) and a < 45:
            cl, cn = small_circle(la, lo, a)
            fig.plot(x=cn, y=cl, pen="0.8p,steelblue@40")
    fig.plot(x=vlon, y=vlat, style="c0.12c", fill="steelblue", pen="0.2p,gray20")
    cl, cn = small_circle(mlat, mlon, a95)
    fig.plot(x=cn, y=cl, pen="3p,firebrick")
    fig.plot(x=[mlon], y=[mlat], style="a0.6c", fill="firebrick", pen="0.5p,black")
    fig.plot(x=[0], y=[90], style="+0.5c", pen="1.5p,black")
fig.show(width=1100)
pygmt.config(FONT_TITLE="18p,Helvetica-Bold,black", FONT_ANNOT_PRIMARY="12p,Helvetica,black")  # restore defaults


### The effect of data quality: one window, three thresholds

Now hold the window fixed (100 Ma) and vary only the quality cut. With Q ≥ 2 the cluster is large but diffuse (overprints and poorly dated poles included); Q ≥ 4 keeps a small, clean population — fewer poles, so A95 can *grow* even though the data are better. This trade-off between data quantity and quality is exactly why the threshold is a free parameter.


In [ ]:
# Cell 7 — VGP cluster at 100 Ma for Q>=2 / Q>=3 / Q>=4
pygmt.config(FONT_TITLE="26p,Helvetica-Bold,black", FONT_ANNOT_PRIMARY="15p,Helvetica,black")
CLUSTER_AGE = 100
fig = pygmt.Figure()
for j, qm in enumerate((2, 3, 4)):
    if j > 0:
        fig.shift_origin(xshift="13.5c")
    w_sel = window_vgps(vgps[qm], CLUSTER_AGE)
    mlat, mlon, a95, kappa, v = mean_vgp(w_sel)
    vlat = np.degrees(np.arcsin(v[:, 2])); vlon = np.degrees(np.arctan2(v[:, 1], v[:, 0]))
    fig.basemap(region="g", projection=f"G{mlon}/{max(mlat, 45)}/12c",
                frame=["g30", f"+tQ>={qm}: n={len(w_sel)}, A95={a95:.1f}\260"])
    fig.coast(land="gray95", shorelines="0.2p,gray60")
    fig.plot(x=vlon, y=vlat, style="c0.12c", fill="steelblue", pen="0.2p,gray20")
    cl, cn = small_circle(mlat, mlon, a95)
    fig.plot(x=cn, y=cl, pen="3p,firebrick")
    fig.plot(x=[mlon], y=[mlat], style="a0.6c", fill="firebrick", pen="0.5p,black")
    fig.plot(x=[0], y=[90], style="+0.5c", pen="1.5p,black")
fig.show(width=1100)
pygmt.config(FONT_TITLE="18p,Helvetica-Bold,black", FONT_ANNOT_PRIMARY="12p,Helvetica,black")  # restore defaults


## 4. The apparent polar wander path in the African plate frame

Running-mean APWPs for each quality threshold, then the preferred path on a north-polar map with A95 circles and age labels, colour-coded with the **batlow** scientific colour scale.


In [ ]:
# Cell 8 — compute APWPs for Q>=2/3/4
def compute_apwp(qm, scheme=SCHEME):
    rows = []
    for age in np.arange(END_AGE, START_AGE + 1, STEP):
        w_sel = window_vgps(vgps[qm], age)
        if len(w_sel) < 3:
            continue
        mlat, mlon, a95, kappa, _ = mean_vgp(w_sel, scheme)
        rows.append(dict(age=age, plat=mlat, plon=mlon, a95=a95, n=len(w_sel)))
    return pd.DataFrame(rows)

apwp_by_q = {qm: compute_apwp(qm) for qm in (2, 3, 4)}
apwp = apwp_by_q[Q_MIN]
apwp.head(8)


In [ ]:
# Cell 9 — APWP map (African plate frame), batlow age colouring, A95 circles, age labels
fig = pygmt.Figure()
fig.basemap(region="g", projection="G0/90/22c",
            frame=["g30", f"+tAPWP in African plate frame (Q>={Q_MIN}, {WINDOW} Myr windows)"])
fig.coast(land="gray95", shorelines="0.2p,gray30")
pygmt.makecpt(cmap="batlow", series=[0, START_AGE])
for _, r in apwp.iterrows():
    if np.isfinite(r.a95) and r.a95 < 60:
        cl, cn = small_circle(r.plat, r.plon, r.a95)
        fig.plot(x=cn, y=cl, pen="0.25p,gray60")
fig.plot(x=apwp.plon, y=apwp.plat, pen="1p,gray40")
fig.plot(x=apwp.plon, y=apwp.plat, style="c0.30c", fill=apwp.age, cmap=True, pen="0.3p,black")
for _, r in apwp.iloc[::3].iterrows():
    fig.text(x=r.plon + 5, y=r.plat + 2, text=f"{r.age:.0f}", font="12p,Helvetica-Bold,black")
fig.colorbar(frame="af+lAge (Ma)")
fig.show(width=1100)


## 5. How robust is the path? Sensitivity, frame differences and APW rate

Three diagnostics:

1. **Quality sensitivity** — great-circle distance between the mean poles of the Q ≥ 3 / Q ≥ 4 paths and the Q ≥ 2 path;
2. **Frame difference** — how far central Africa would be displaced at each age if reconstructed with the Q ≥ 3 or Q ≥ 4 frame instead of the Q ≥ 2 frame (the same information expressed where it matters: on the map);
3. **APW rate** along the preferred path — peaks are candidate true-polar-wander episodes.


In [ ]:
# Cell 10 — sensitivity, frame difference and APW rate charts
def gcd_deg(lat1, lon1, lat2, lon2):
    return math.degrees(math.acos(np.clip(np.dot(xyz([lat1],[lon1])[0], xyz([lat2],[lon2])[0]), -1, 1)))

def frame_rotation_for(row):  # minimal rotation bringing mean pole -> north pole
    p = xyz([row.plat], [row.plon])[0]; north = np.array([0, 0, 1.0])
    ax = np.cross(p, north)
    if np.linalg.norm(ax) < 1e-9:
        return pygplates.FiniteRotation()
    ax /= np.linalg.norm(ax)
    ang = math.acos(np.clip(p @ north, -1, 1))
    return pygplates.FiniteRotation(
        pygplates.PointOnSphere(math.degrees(math.asin(ax[2])),
                                math.degrees(math.atan2(ax[1], ax[0]))), ang)

AFR_POINT = pygplates.PointOnSphere(0, 20)   # central Africa
idx = {qm: apwp_by_q[qm].set_index("age") for qm in (2, 3, 4)}

# Compute all series first so each panel's y-range can be fitted to its data.
gcd_series, disp_series = {}, {}
for qm in (3, 4):
    common = idx[2].index.intersection(idx[qm].index)
    gcd_series[qm] = (list(common),
                      [gcd_deg(idx[2].loc[t].plat, idx[2].loc[t].plon,
                               idx[qm].loc[t].plat, idx[qm].loc[t].plon) for t in common])
    ages_d, d = [], []
    for t in common:
        p1 = frame_rotation_for(idx[2].loc[t]) * AFR_POINT
        p2 = frame_rotation_for(idx[qm].loc[t]) * AFR_POINT
        d.append(math.degrees(pygplates.GeometryOnSphere.distance(p1, p2))); ages_d.append(t)
    disp_series[qm] = (ages_d, d)
path = apwp.reset_index(drop=True)
mid_ages = 0.5 * (path.age.values[1:] + path.age.values[:-1])
rates = [gcd_deg(path.plat[i], path.plon[i], path.plat[i+1], path.plon[i+1]) /
         max(1e-6, path.age[i+1] - path.age[i]) for i in range(len(path) - 1)]

y_gcd  = 1.1 * max(max(v[1]) for v in gcd_series.values())
y_disp = 1.1 * max(max(v[1]) for v in disp_series.values())
y_rate = 1.1 * max(rates)

fig = pygmt.Figure()
# Panel 1: GCD between APWPs
fig.basemap(region=[0, START_AGE, 0, y_gcd], projection="X24c/6.5c",
            frame=["xaf+lAge (Ma)", "yaf+lGCD (\260)", "+tAPWP sensitivity to the quality threshold (vs Q>=2)"])
for qm, pen in ((3, "1.5p,dodgerblue4"), (4, "1.5p,firebrick")):
    fig.plot(x=gcd_series[qm][0], y=gcd_series[qm][1], pen=pen, label=f"Q>={qm} vs Q>=2")
fig.legend(position="JTL+jTL+o0.3c", box="+gwhite+p0.5p")
# Panel 2: displacement of central Africa between frames
fig.shift_origin(yshift="-8.8c")
fig.basemap(region=[0, START_AGE, 0, y_disp], projection="X24c/6.5c",
            frame=["xaf+lAge (Ma)", "yaf+lDisplacement (\260)", "+tFrame difference: displacement of central Africa (vs Q>=2 frame)"])
for qm, pen in ((3, "1.5p,dodgerblue4"), (4, "1.5p,firebrick")):
    fig.plot(x=disp_series[qm][0], y=disp_series[qm][1], pen=pen, label=f"Q>={qm} vs Q>=2")
fig.legend(position="JTL+jTL+o0.3c", box="+gwhite+p0.5p")
# Panel 3: APW rate along the preferred path
fig.shift_origin(yshift="-8.8c")
fig.basemap(region=[0, START_AGE, 0, y_rate], projection="X24c/6.5c",
            frame=["xaf+lAge (Ma)", "yaf+lAPW rate (\260/Myr)", f"+tAPW rate along the Q>={Q_MIN} path"])
fig.plot(x=mid_ages, y=rates, pen="1.5p,gray30")
fig.show(width=1100)


## 6. Convert the APWP to absolute rotations and embed as anchor plate 701702

The absolute rotation at each age is the minimal (great-circle) rotation bringing the mean pole to the geographic north pole — the zero-paleolongitude convention. We embed the frame the same way the model's own paleomagnetic frame (`701701`) is implemented: a virtual plate **701702** moving relative to 701, holding the *inverse* rotations, so that `anchor_plate_id=701702` reconstructs everything in the new frame.


In [ ]:
# Cell 11 — write the augmented rotation file
import shutil

rot_lines = []
for _, r in apwp.sort_values("age").iterrows():
    p = xyz([r.plat], [r.plon])[0]; north = np.array([0, 0, 1.0])
    ax = np.cross(p, north)
    if np.linalg.norm(ax) < 1e-9:
        lat_a = lon_a = ang = 0.0
    else:
        ax /= np.linalg.norm(ax)
        ang = math.degrees(math.acos(np.clip(p @ north, -1, 1)))
        lat_a, lon_a = math.degrees(math.asin(ax[2])), math.degrees(math.atan2(ax[1], ax[0]))
    rot_lines.append(f"701702 {r.age:7.2f} {lat_a:7.2f} {lon_a:8.2f} {-ang:8.3f}  701 ! GPMDB pmag frame Q>={Q_MIN} A95={r.a95:.1f}")
if not rot_lines or " 0.00 " not in rot_lines[0]:
    rot_lines.insert(0, "701702    0.00    0.00     0.00    0.000  701 ! GPMDB pmag frame, present day")

rm_files = model.get_rotation_model()
src_rot = rm_files[0] if isinstance(rm_files, list) else rm_files
aug_rot = "data/Zahirovic2022_with_gpmdb_frame.rot"
shutil.copy(src_rot, aug_rot)
with open(aug_rot, "a") as f:
    f.write("\n" + "\n".join(rot_lines) + "\n")
print(f"Wrote {aug_rot} with {len(rot_lines)} rotations for anchor plate 701702")

rm_aug = pygplates.RotationModel(aug_rot)
print("0 Ma check (should be ~identity):", rm_aug.get_rotation(0.0, 701, anchor_plate_id=701702))


## Extend this

- Vary `Q_MIN`, `WINDOW` and `SCHEME` and overlay the resulting APWPs — how robust is the path in the Paleozoic versus the Mesozoic?
- Replace the running mean with a spherical smoothing spline (Jupp & Kent, 1987) weighted by Q.
- Export a fresh CSV from the internet-based GPMDB (gpmdb.net) — ~800 more poles and curated Q factors — and re-run.
- Compare your path with the global APWP of Vaes et al. (2023) for 0–320 Ma.
- Continue to **T17**, where this frame is compared against the model's built-in paleomagnetic frame in reconstructions and an animation.
